In [8]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import Gemma3TextModel, AutoTokenizer

embeddinggemma = SentenceTransformer("google/embeddinggemma-300m")
embeddinggemma.save_pretrained("magibu-200m-base")

model = Gemma3TextModel.from_pretrained("google/embeddinggemma-300m")

embeddings = model.embed_tokens.weight
print(f"Embeddings shape: {embeddings.shape}")

org_tokenizer = AutoTokenizer.from_pretrained("google/embeddinggemma-300m")
target_tokenizer = AutoTokenizer.from_pretrained("alibayram/magibu-128k-processor")
print(f"Tokenizer vocab size: {org_tokenizer.vocab_size}, {target_tokenizer.vocab_size}")


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Embeddings shape: torch.Size([262144, 768])
Tokenizer vocab size: 262144, 131072


In [ ]:

errors = []
for i in range(target_tokenizer.vocab_size):
    if i not in token_id_map or not token_id_map[i]:
        errors.append(i)
        # Initialize with zeros for unmapped tokens
        continue

    source_ids = token_id_map[i]
    # MEAN token strategy: average the source token embeddings
    # remove <bos> if present
    if source_ids[0] == org_tokenizer.bos_token_id:
        source_ids = source_ids[1:]
    embeddings_to_average = embeddings[source_ids]
    new_embeddings[i] = embeddings_to_average.mean(dim=0)

    if (i + 1) % 10000 == 0:
        print(f"Mapped {i + 1}/{target_tokenizer.vocab_size} embeddings", flush=True)

model.resize_token_embeddings(embeddings.shape[1])
print(model)

In [9]:
new_embeddings = torch.zeros(target_tokenizer.vocab_size, embeddings.shape[1], dtype=embeddings.dtype)
print(f"Embeddings {new_embeddings.shape}")

Embeddings torch.Size([131072, 768])


In [10]:
source_vocab = org_tokenizer.get_vocab()
target_vocab = target_tokenizer.get_vocab()

token_id_map = {}
direct_matches = 0
tokenized_matches = 0

for token_str, target_id in target_vocab.items():
    if token_str in source_vocab:
        # Direct match - use source token ID directly
        token_id_map[target_id] = [source_vocab[token_str]]
        direct_matches += 1
    else:
        # Token not in source vocab - need to tokenize
        encoded = org_tokenizer.encode(token_str, add_special_tokens=False)
        if encoded:
            token_id_map[target_id] = encoded
            tokenized_matches += 1

print(f"Token id map {len(token_id_map)}")

Token id map 131073


In [11]:
embeddings.shape

torch.Size([262144, 768])

In [1]:
from sentence_transformers import SentenceTransformer

embeddinggemma = SentenceTransformer("google/embeddinggemma-300m")
embeddinggemma

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 2048, 'do_lower_case': False, 'architecture': 'Gemma3TextModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 3072, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
  (3): Dense({'in_features': 3072, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
  (4): Normalize()
)

In [4]:
embeddinggemma.save("embeddinggemma-300m")

In [3]:
from transformers import Gemma3TextModel

model = Gemma3TextModel.from_pretrained("embeddinggemma-300m")
model

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Gemma3TextModel(
  (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 768, padding_idx=0)
  (layers): ModuleList(
    (0-23): 24 x Gemma3DecoderLayer(
      (self_attn): Gemma3Attention(
        (q_proj): Linear(in_features=768, out_features=768, bias=False)
        (k_proj): Linear(in_features=768, out_features=256, bias=False)
        (v_proj): Linear(in_features=768, out_features=256, bias=False)
        (o_proj): Linear(in_features=768, out_features=768, bias=False)
        (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
        (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
      )
      (mlp): Gemma3MLP(
        (gate_proj): Linear(in_features=768, out_features=1152, bias=False)
        (up_proj): Linear(in_features=768, out_features=1152, bias=False)
        (down_proj): Linear(in_features=1152, out_features=768, bias=False)
        (act_fn): GELUTanh()
      )
      (input_layernorm): Gemma3RMSNorm((768,), eps=1e-06)
      (post_attention_layernorm): Gemma3RMSNorm((768,), eps=1e-06)


In [6]:
embeddings = model.model.embed_tokens.weight
print(f"Embeddings shape: {embeddings.shape}")

Embeddings shape: torch.Size([262144, 768])


In [2]:
from sentence_transformers import SentenceTransformer

embeddingmagibu = SentenceTransformer("magibu-200m-base")
embeddingmagibu

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Gemma3TextModel LOAD REPORT from: magibu-200m-base
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer(
  (0): Transformer({'max_seq_length': 2048, 'do_lower_case': False, 'architecture': 'Gemma3TextModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 3072, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
  (3): Dense({'in_features': 3072, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
  (4): Normalize()
)

In [1]:
from transformers import AutoTokenizer

org_tokenizer = AutoTokenizer.from_pretrained("magibu-200m-base")
org_tokenizer.push_to_hub("alibayram/magibu-128k-processor")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/alibayram/magibu-128k-processor/commit/030c5431123132a6c1fa27d08b3eb400a836a42a', commit_message='Upload tokenizer', commit_description='', oid='030c5431123132a6c1fa27d08b3eb400a836a42a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/alibayram/magibu-128k-processor', endpoint='https://huggingface.co', repo_type='model', repo_id='alibayram/magibu-128k-processor'), pr_revision=None, pr_num=None)

In [11]:
from transformers import AutoTokenizer

org_tokenizer = AutoTokenizer.from_pretrained("embeddinggemma-300m")
target_tokenizer = AutoTokenizer.from_pretrained("alibayram/magibu-128k-processor", use_fast=True)
print(f"Tokenizer vocab size: {org_tokenizer.vocab_size}, {target_tokenizer.vocab_size}")

Tokenizer vocab size: 262144, 131072


In [12]:
source_vocab = org_tokenizer.get_vocab()
target_vocab = target_tokenizer.get_vocab()

token_id_map = {}
direct_matches = 0
tokenized_matches = 0

for token_str, target_id in target_vocab.items():
    if token_str in source_vocab:
        # Direct match - use source token ID directly
        token_id_map[target_id] = [source_vocab[token_str]]
        direct_matches += 1
    else:
        # Token not in source vocab - need to tokenize
        encoded = org_tokenizer.encode(token_str, add_special_tokens=False)
        if encoded:
            token_id_map[target_id] = encoded
            tokenized_matches += 1

In [15]:
import torch

new_embeddings = torch.zeros(target_tokenizer.vocab_size, embeddings.shape[1], dtype=embeddings.dtype)
errors = []
for i in range(target_tokenizer.vocab_size):
    if i not in token_id_map or not token_id_map[i]:
        errors.append(i)
        # Initialize with zeros for unmapped tokens
        continue
    
    source_ids = token_id_map[i]
    # MEAN token strategy: average the source token embeddings
    # remove <bos> if present
    if source_ids[0] == org_tokenizer.bos_token_id:
        source_ids = source_ids[1:]
    embeddings_to_average = embeddings[source_ids]
    new_embeddings[i] = embeddings_to_average.mean(dim=0)
    
    if (i + 1) % 10000 == 0:
        print(f"   Mapped {i + 1}/{target_tokenizer.vocab_size} embeddings", flush=True)

   Mapped 10000/131072 embeddings
   Mapped 20000/131072 embeddings
   Mapped 30000/131072 embeddings
   Mapped 40000/131072 embeddings
   Mapped 50000/131072 embeddings
   Mapped 60000/131072 embeddings
   Mapped 70000/131072 embeddings
   Mapped 80000/131072 embeddings
   Mapped 90000/131072 embeddings
   Mapped 100000/131072 embeddings
   Mapped 110000/131072 embeddings
   Mapped 120000/131072 embeddings
   Mapped 130000/131072 embeddings


In [28]:
model.resize_token_embeddings(embeddings.shape[1])
model.model.embed_tokens.weight = torch.nn.Parameter(new_embeddings)
print(f"Completed embedding transfer with {len(errors)} errors.")

model = model.to(torch.bfloat16)

model.save_pretrained("magibu-200m-base")
target_tokenizer.save_pretrained("magibu-200m-base")


Completed embedding transfer with 0 errors.


('magibu-200m-base/tokenizer_config.json',
 'magibu-200m-base/chat_template.jinja',
 'magibu-200m-base/tokenizer.json')

In [29]:
model.push_to_hub("alibayram/magibu-200m-base")
target_tokenizer.push_to_hub("alibayram/magibu-200m-base")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/alibayram/magibu-200m-base/commit/2263e9729ccc5bc3410b866116709161e79ca4d9', commit_message='Upload tokenizer', commit_description='', oid='2263e9729ccc5bc3410b866116709161e79ca4d9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/alibayram/magibu-200m-base', endpoint='https://huggingface.co', repo_type='model', repo_id='alibayram/magibu-200m-base'), pr_revision=None, pr_num=None)

In [19]:
parameter_count = sum(p.numel() for p in model.parameters())
print(f"Model parameter count: {parameter_count / 1_000_000:.2f}M")

Model parameter count: 202.79M
